<span style="font-size: 24px;">Ejercicio 1. Creación de una bases de datos de películas extraídas de un API </span>

In [1]:
# Importo las librerías que voy a necesitar

import requests  # Para llamar a las APIs
import pandas as pd  # Para poder convertir los datos en df
import mysql.connector  # Para conectar Python con MySQL
from mysql.connector import Error  # Para importar los errores de SQL
import os   # Para crear variables de entorno y poder mantener la contraseña secreta
from dotenv import load_dotenv  # Para leer el archivo .env de la variable de entorno
load_dotenv()  # Carga el contenido del .env
password_sql = os.getenv("pass_sql") # Para acceder a la contraseña que hemos indicado en el .env
import numpy as np  # Para poder hacer el clean

<span style="font-size: 20px;">Fase 1: Extracción de datos de películas</span>
 
El objetivo es extraer 100 películas de esta API utilizando el siguiente endpoint:
https://beta.adalab.es/resources/apis/pelis/pelis.json
Datos a extraer: título, año de lanzamiento, duración (en minutos), género, contenidon para adultos (sí o no).

In [2]:
url = "https://beta.adalab.es/resources/apis/pelis/pelis.json"

datos = requests.get(url)   # Hacemos la petición a la API

In [3]:
datos.status_code  # Confirmamos que la petición ha sido exitosa

200

In [4]:
datos_pelis = datos.json()  # Pasamos los datos a formato json para poder leerlos bien en Python
df_pelis = pd.DataFrame(datos_pelis)  # Convertimos esos datos en tabla (DataFrame) con ayuda de Pandas
df_pelis

,id,titulo,año,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,False,"[es, en]"
1,2,The Godfather Part II,1974,202,Crimen,False,"[es, en]"
2,3,Pulp Fiction,1994,154,Crimen,True,"[es, en]"
3,4,Forrest Gump,1994,142,Drama,False,"[es, en, fr]"
4,5,The Dark Knight,2008,152,Acción,False,"[es, en]"
...,...,...,...,...,...,...,...
95,96,La vita è bella,1997,116,Drama,False,"[es, en, it]"
96,97,Requiem for a Dream,2000,102,Drama,True,"[es, en]"
97,98,Memento,2000,113,Thriller,True,"[es, en]"
98,99,Eternal Sunshine of the Spotless Mind,2004,108,Drama,False,"[es, en]"


<span style="font-size: 20px;">Fase 2: Creación de la Base de Datos</span>

Podemos elegir entre crear la base de datos en MySQLWorkbench o en Python, yo voy a elegir utilizar Python.

In [5]:
# Nos conectamos con MySQL

try:
    connection = mysql.connector.connect (
    host = "127.0.0.1", 
    user = "root",
    password = password_sql, 
    )
    print ("Conexion exitosa")
except Error as e:
    print ("Ha ocurrido un error: {e}")

Conexion exitosa


In [6]:
# Vamos a crear el cursor y la base de datos

try:
    cursor = connection.cursor()
    query_crear_bbdd = "CREATE DATABASE IF NOT EXISTS pelis"
    cursor.execute(query_crear_bbdd)
    print("Query existosa")
except Error as e:
    print(e)

Query existosa


In [7]:
# Queremos saber qué tipo de datos tiene la tabla importada para poder crear nuestra tabla en la base de datos

df_pelis.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          100 non-null    int64 
 1   titulo      100 non-null    str   
 2   año         100 non-null    int64 
 3   duracion    100 non-null    int64 
 4   genero      100 non-null    str   
 5   adultos     100 non-null    bool  
 6   subtitulos  100 non-null    object
dtypes: bool(1), int64(3), object(1), str(2)
memory usage: 4.9+ KB


In [8]:
# Le pedimos que use la base de datos que acabamos de crear y creamos la tabla

cursor.execute("use pelis")
query_borrar = '''drop table pelis'''
cursor.execute(query_borrar)
connection.commit()
query_crear_tabla = '''CREATE TABLE pelis (
                        id INT PRIMARY KEY AUTO_INCREMENT, 
                        titulo VARCHAR(200) NOT NULL,
                        año YEAR,
                        duracion INT,
                        genero VARCHAR(100),
                        adultos BOOL,
                        subtitulos VARCHAR(50)
                                               );'''
cursor.execute(query_crear_tabla)
print ("Tabla creada correctamente")

Tabla creada correctamente


<span style="font-size: 20px;">Fase 3: Inserción de los Datos en la Base de Datos</span>

En este apartado vamos a realizar la inserción de los datos extraídos en las tablas creadas en nuestra base de datos MySQL.

In [9]:
# Vamos a ver si hay alguna celda vacía

df_pelis.isnull().sum()

id            0
titulo        0
año           0
duracion      0
genero        0
adultos       0
subtitulos    0
dtype: int64

In [10]:
# Necesito convertir subtitulos en un string, porque ahora es una lista

df_pelis['subtitulos'] = df_pelis['subtitulos'].astype(str)

In [11]:
# Añadimos los datos que hemos importado de la API a la tabla de nuestra base de datos

try:
    query_insertar = '''INSERT INTO pelis (id, titulo, año, duracion, genero, adultos, subtitulos)
                        VALUES (%s, %s, %s, %s, %s, %s, %s)'''
    values = df_pelis.values.tolist() 
    cursor.executemany(query_insertar, values)
    connection.commit()
    print(f"Se han insertado nuevos {cursor.rowcount} valores")
except Error as e:
    print(e)

Se han insertado nuevos 100 valores


<span style="font-size: 20px;">Fase 4: Obtener información a partir de los datos</span>

Una vez que tenemos toda la información, vamos a responder las siguientes preguntas utilizando consultas en SQL.

<span style="font-size: 18px;">- ¿Cuántas películas tienen una duración superior a 120 minutos?</span>

In [12]:
query_pelis_con_mas_de_120_mins = '''SELECT COUNT(*) AS pelis_con_mas_de_120_mins 
FROM pelis 
WHERE duracion > 120 
ORDER BY duracion DESC;'''
df_pelis_con_mas_de_120_mins = pd.read_sql(query_pelis_con_mas_de_120_mins, connection)
df_pelis_con_mas_de_120_mins


C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\3378978539.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pelis_con_mas_de_120_mins = pd.read_sql(query_pelis_con_mas_de_120_mins, connection)


,pelis_con_mas_de_120_mins
0,59


<span style="font-size: 18px;">- ¿Cuántas películas incluyen subtítulos en español?</span>

In [13]:
query_pelis_con_subtitulos_en_español = '''SELECT COUNT(*) AS pelis_con_subtitulos_en_español 
FROM pelis 
WHERE subtitulos LIKE "%es%";'''
df_pelis_con_subtitulos_en_español = pd.read_sql(query_pelis_con_subtitulos_en_español, connection)
df_pelis_con_subtitulos_en_español

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\445771845.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pelis_con_subtitulos_en_español = pd.read_sql(query_pelis_con_subtitulos_en_español, connection)


,pelis_con_subtitulos_en_español
0,100


<span style="font-size: 18px;">- ¿Cuántas películas tienen contenido adulto?</span>

In [14]:
query_pelis_contenido_adulto = '''SELECT COUNT(*) AS pelis_contenido_adulto 
FROM pelis 
WHERE adultos = 1;'''
df_pelis_contenido_adulto = pd.read_sql(query_pelis_contenido_adulto, connection)
df_pelis_contenido_adulto

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\694473635.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pelis_contenido_adulto = pd.read_sql(query_pelis_contenido_adulto, connection)


,pelis_contenido_adulto
0,47


<span style="font-size: 18px;">- ¿Cuál es la película más antigua registrada en la base de datos?</span>

In [16]:
query_peli_mas_antigua = '''SELECT titulo AS peli_mas_antigua, año
FROM pelis
ORDER BY año ASC 
LIMIT 1;'''
df_peli_mas_antigua = pd.read_sql(query_peli_mas_antigua, connection)
df_peli_mas_antigua

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\1258888803.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_peli_mas_antigua = pd.read_sql(query_peli_mas_antigua, connection)


,peli_mas_antigua,año
0,Citizen Kane,1941


<span style="font-size: 18px;">- Muestra el promedio de duración de las películas agrupado por género.</span>

In [17]:
query_promedio_duracion_por_genero = '''SELECT genero, ROUND(AVG(duracion), 0) AS promedio_duracion_por_genero 
FROM pelis 
GROUP BY genero;'''
df_promedio_duracion_por_genero = pd.read_sql(query_promedio_duracion_por_genero, connection)
df_promedio_duracion_por_genero

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\2361697455.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_promedio_duracion_por_genero = pd.read_sql(query_promedio_duracion_por_genero, connection)


,genero,promedio_duracion_por_genero
0,Crimen,154.0
1,Drama,126.0
2,Acción,139.0
3,Ciencia ficción,136.0
4,Romance,140.0
5,Bélico,161.0
6,Thriller,122.0
7,Musical,128.0
8,Fantasía,160.0
9,Aventura,133.0


<span style="font-size: 18px;">- ¿Cuántas películas por año se han registrado en la base de datos? Ordena de mayor a menor.</span>

In [19]:
query_pelis_por_año = '''SELECT año, COUNT(*) AS pelis_por_año
FROM pelis
GROUP BY año
ORDER BY pelis_por_año desc;'''
df_pelis_por_año = pd.read_sql(query_pelis_por_año, connection)
df_pelis_por_año

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\3282154765.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pelis_por_año = pd.read_sql(query_pelis_por_año, connection)


,año,pelis_por_año
0,2001,5
1,2013,4
2,1994,4
3,2008,4
4,1999,4
5,2017,4
6,2010,3
7,1998,3
8,2014,3
9,2000,3


<span style="font-size: 18px;">- ¿Cuál es el año con más películas en la base de datos?</span>

In [20]:
query_año_con_mas_pelis = '''SELECT año AS año_con_mas_pelis, COUNT(*) AS año
FROM pelis
GROUP BY año
ORDER BY año desc
LIMIT 1;'''
df_año_con_mas_pelis = pd.read_sql(query_año_con_mas_pelis, connection)
df_año_con_mas_pelis

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\3560252924.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_año_con_mas_pelis = pd.read_sql(query_año_con_mas_pelis, connection)


,año_con_mas_pelis,año
0,2001,5


<span style="font-size: 18px;">- Obtén un listado de todos los géneros y cuántas películas corresponden a cada uno.</span>

In [21]:
query_cantidad_pelis = '''SELECT genero, COUNT(*) AS cantidad_pelis
FROM pelis
GROUP BY genero
ORDER BY cantidad_pelis DESC;'''
df_cantidad_pelis = pd.read_sql(query_cantidad_pelis, connection)
df_cantidad_pelis

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\3377762508.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cantidad_pelis = pd.read_sql(query_cantidad_pelis, connection)


,genero,cantidad_pelis
0,Drama,27
1,Ciencia ficción,13
2,Acción,9
3,Animación,9
4,Crimen,7
5,Terror,7
6,Thriller,6
7,Fantasía,5
8,Romance,3
9,Aventura,3


<span style="font-size: 18px;">- Muestra todas las películas cuyo título contenga la palabra "Godfather" (puedes usar cualquier
palabra).</span>

In [22]:
query_Godfather = '''SELECT * 
FROM pelis
WHERE titulo LIKE "%Godfather%";'''
df_Godfather = pd.read_sql(query_Godfather, connection)
df_Godfather

C:\Users\Usuario\AppData\Local\Temp\ipykernel_28724\302262098.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_Godfather = pd.read_sql(query_Godfather, connection)


,id,titulo,año,duracion,genero,adultos,subtitulos
0,1,The Godfather,1972,175,Crimen,0,"['es', 'en']"
1,2,The Godfather Part II,1974,202,Crimen,0,"['es', 'en']"
